In [1]:
# ==========================================
# CELL 1: IMPORTS
# ==========================================
import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

print("All imports loaded!")
print("TensorFlow:", tf.__version__)

2026-04-30 17:36:46.785840: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777550806.824555 1462597 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777550806.837515 1462597 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777550806.969153 1462597 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777550806.969189 1462597 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777550806.969193 1462597 computation_placer.cc:177] computation placer alr

All imports loaded!
TensorFlow: 2.19.1


In [2]:
# ==========================================
# CELL 2: REPRODUCIBILITY
# ==========================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print(f"Seed {SEED} fixed.")

Seed 42 fixed.


In [3]:
# ==========================================
# CELL 3: GPU SETUP
# ==========================================
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print("GPU setup warning:", e)
else:
    print("No GPU found, running on CPU.")

GPU memory growth enabled.


In [4]:
# ==========================================
# CELL 4: MIXED PRECISION
# ==========================================
from tensorflow.keras.mixed_precision import set_global_policy
set_global_policy("mixed_float16")
print("Mixed precision enabled.")

Mixed precision enabled.


In [77]:
# ==========================================
# CELL 5: CONFIGURATION
# ==========================================
student_img_size = 32
teacher_img_size = 224
batch_size = 32
epochs = 20

train_path = "/home/22EC1102/soumen/satarupa/hydrophobicity/Hydrophobicity Classes Photos/train"
val_path = "/home/22EC1102/soumen/satarupa/hydrophobicity/Hydrophobicity Classes Photos/validation"

teacher_model_path = "/home/22EC1102/soumen/satarupa/hydrophobicity/teacher_model_mobilenetv3.keras"
baseline_model_path = "hydro_baseline_M5.keras"
kd_model_path = "hydro_kd_M5.keras"

num_classes = 7

print("Student size:", student_img_size)
print("Teacher size:", teacher_img_size)
print("Batch size:", batch_size)
print("Classes:", num_classes)

Student size: 32
Teacher size: 224
Batch size: 32
Classes: 7


In [78]:
# ==========================================
# CELL 6: HYDROPHOBICITY DATA PREPROCESSING
# same style as teacher_model_hydrophobisity.ipynb
# ==========================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data_teacher224 = train_datagen.flow_from_directory(
    train_path,
    target_size=(teacher_img_size, teacher_img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_data_teacher224 = val_datagen.flow_from_directory(
    val_path,
    target_size=(teacher_img_size, teacher_img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

print("Class indices:", train_data_teacher224.class_indices)
class_names = list(train_data_teacher224.class_indices.keys())
num_classes = len(class_names)
print("Detected classes:", class_names)
print("Num classes:", num_classes)

Found 2800 images belonging to 7 classes.
Found 700 images belonging to 7 classes.
Class indices: {'HC1': 0, 'HC2': 1, 'HC3': 2, 'HC4': 3, 'HC5': 4, 'HC6': 5, 'HC7': 6}
Detected classes: ['HC1', 'HC2', 'HC3', 'HC4', 'HC5', 'HC6', 'HC7']
Num classes: 7


In [79]:
# ==========================================
# CELL 7: STUDENT DATA GENERATORS (32x32)
# same hydrophobicity dataset, smaller size for student
# ==========================================
train_data_student32 = train_datagen.flow_from_directory(
    train_path,
    target_size=(student_img_size, student_img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_data_student32 = val_datagen.flow_from_directory(
    val_path,
    target_size=(student_img_size, student_img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

print("Student train batches:", len(train_data_student32))
print("Student val batches:", len(val_data_student32))

Found 2800 images belonging to 7 classes.
Found 700 images belonging to 7 classes.
Student train batches: 88
Student val batches: 22


In [80]:
# ==========================================
# CELL 8: OPTIONAL SHAPE CHECK
# ==========================================
images32, labels32 = next(train_data_student32)
images224, labels224 = next(train_data_teacher224)

print("Student batch image shape:", images32.shape)
print("Student batch label shape:", labels32.shape)
print("Teacher batch image shape:", images224.shape)
print("Teacher batch label shape:", labels224.shape)

Student batch image shape: (32, 32, 32, 3)
Student batch label shape: (32, 7)
Teacher batch image shape: (32, 224, 224, 3)
Teacher batch label shape: (32, 7)


In [81]:
# ==========================================
# CELL 9: CUSTOM STUDENT MODEL
# MobileNetV3-style student with tunable hyperparameters
# ==========================================
def create_mobilenetv3_student(
    conv_layers,
    filters,
    kernel_size,
    fc_layers,
    use_bn,
    use_dropout,
    input_shape=(32, 32, 3),
    num_classes=7,
    expand_ratio=4
):
    def inverted_residual_block(x, out_channels, ksize, stride, use_se=True):
        in_channels = x.shape[-1]
        x_input = x
        hidden_dim = int(in_channels * expand_ratio)

        if expand_ratio != 1:
            x_expanded = layers.Conv2D(hidden_dim, 1, padding="same", use_bias=False)(x)
            if use_bn:
                x_expanded = layers.BatchNormalization()(x_expanded)
            x_expanded = layers.ReLU(max_value=6.0)(x_expanded)
        else:
            x_expanded = x

        x = layers.DepthwiseConv2D(ksize, strides=stride, padding="same", use_bias=False)(x_expanded)
        if use_bn:
            x = layers.BatchNormalization()(x)
        x = layers.ReLU(max_value=6.0)(x)

        if use_se and hidden_dim > 0:
            se_channels = max(1, int(hidden_dim * 0.25))
            se = layers.GlobalAveragePooling2D()(x)
            se = layers.Dense(se_channels, activation="relu")(se)
            se = layers.Dense(hidden_dim, activation="hard_sigmoid")(se)
            se = layers.Reshape((1, 1, hidden_dim))(se)
            x = layers.Multiply()([x, se])

        x = layers.Conv2D(out_channels, 1, padding="same", use_bias=False)(x)
        if use_bn:
            x = layers.BatchNormalization()(x)

        if stride == 1 and in_channels == out_channels:
            x = layers.Add()([x, x_input])

        return x

    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(filters, 3, strides=(1, 1), padding="same", use_bias=False)(inputs)
    if use_bn:
        x = layers.BatchNormalization()(x)
    x = layers.ReLU(max_value=6.0)(x)

    curr_filters = filters
    for i in range(conv_layers):
        next_filters = min(curr_filters * 2, 256)
        stride = 2 if i < 2 else 1
        x = inverted_residual_block(x, next_filters, kernel_size, stride)
        x = inverted_residual_block(x, next_filters, kernel_size, 1)
        curr_filters = next_filters

    x = layers.GlobalAveragePooling2D()(x)

    if fc_layers == 4:
        x = layers.Dense(512, activation="relu")(x)
        if use_dropout:
            x = layers.Dropout(0.5)(x)
    elif fc_layers == 3:
        x = layers.Dense(256, activation="relu")(x)
        if use_dropout:
            x = layers.Dropout(0.5)(x)
    elif fc_layers == 2:
        x = layers.Dense(128, activation="relu")(x)
        if use_dropout:
            x = layers.Dropout(0.3)(x)
    elif fc_layers == 1:
        x = layers.Dense(64, activation="relu")(x)
        if use_dropout:
            x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = Model(inputs=inputs, outputs=outputs, name="student_mobilenetv3_hydrophobicity")
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
        jit_compile=True
    )
    return model

In [82]:
# ==========================================
# CELL 10: STUDENT HYPERPARAMETERS
# ==========================================
baseline_params = {
    "conv_layers": 4,
    "filters": 32,
    "kernel_size": 5,
    "fc_layers": 1,
    "use_bn": True,
    "use_dropout": False
}

print("Baseline/student params:", baseline_params)

Baseline/student params: {'conv_layers': 4, 'filters': 32, 'kernel_size': 5, 'fc_layers': 1, 'use_bn': True, 'use_dropout': False}


In [83]:
# ==========================================
# CELL 11: CALLBACKS FOR BASELINE
# ==========================================
checkpoint_baseline = ModelCheckpoint(
    baseline_model_path,
    save_best_only=True,
    monitor="val_accuracy",
    mode="max",
    verbose=1
)

lr_reducer = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True,
    verbose=1
)

In [84]:
# ==========================================
# CELL 12: BUILD BASELINE STUDENT
# ==========================================
student_baseline = create_mobilenetv3_student(
    input_shape=(student_img_size, student_img_size, 3),
    num_classes=num_classes,
    **baseline_params
)

student_baseline.summary()

Model: "student_mobilenetv3_hydrophobicity"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_40 (Conv2D)  │ (None, 32, 32,    │        864 │ input_layer_8[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ conv2d_40[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_38 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_41 (Conv2D)  │ (None, 32, 32,    │      4,096 │ re_lu_38[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_41[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_39 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_16 │ (None, 16, 16,    │      3,200 │ re_lu_39[0][0]    │
│ (DepthwiseConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_40 (ReLU)     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ re_lu_40[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_44 (Dense)    │ (None, 32)        │      4,128 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_45 (Dense)    │ (None, 128)       │      4,224 │ dense_44[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_16          │ (None, 1, 1, 128) │          0 │ dense_45[0][0]    │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_16         │ (None, 16, 16,    │          0 │ re_lu_40[0][0],   │
│ (Multiply)          │ 128)              │            │ reshape_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_42 (Conv2D)  │ (None, 16, 16,    │      8,192 │ multiply_16[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_42[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,089,223 (15.60 MB)

 Trainable params: 4,067,399 (15.52 MB)

 Non-trainable params: 21,824 (85.25 KB)

In [85]:
# ==========================================
# CELL 13: TRAIN BASELINE STUDENT
# ==========================================
print("Training baseline student on hydrophobicity 32x32...")

history_baseline = student_baseline.fit(
    train_data_student32,
    validation_data=val_data_student32,
    epochs=epochs,
    callbacks=[early_stop, checkpoint_baseline, lr_reducer],
    verbose=2
)

student_baseline.save(baseline_model_path)
print("Baseline student saved.")

Training baseline student on hydrophobicity 32x32...
Epoch 1/20


2026-04-30 18:27:30.411923: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_broadcast_multiply_reduce_fusion', 4 bytes spill stores, 4 bytes spill loads




Epoch 1: val_accuracy improved from None to 0.14286, saving model to hydro_baseline_M5.keras
88/88 - 256s - 3s/step - accuracy: 0.5182 - loss: 1.3241 - val_accuracy: 0.1429 - val_loss: 2.0435 - learning_rate: 0.0010
Epoch 2/20

Epoch 2: val_accuracy did not improve from 0.14286
88/88 - 7s - 77ms/step - accuracy: 0.6782 - loss: 0.8327 - val_accuracy: 0.1429 - val_loss: 2.6867 - learning_rate: 0.0010
Epoch 3/20

Epoch 3: val_accuracy did not improve from 0.14286
88/88 - 7s - 76ms/step - accuracy: 0.7189 - loss: 0.6943 - val_accuracy: 0.1429 - val_loss: 2.7986 - learning_rate: 0.0010
Epoch 4/20

Epoch 4: val_accuracy did not improve from 0.14286

Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
88/88 - 7s - 75ms/step - accuracy: 0.7614 - loss: 0.5970 - val_accuracy: 0.1429 - val_loss: 3.3425 - learning_rate: 0.0010
Epoch 5/20

Epoch 5: val_accuracy improved from 0.14286 to 0.18857, saving model to hydro_baseline_M5.keras
88/88 - 7s - 85ms/step - accuracy: 0.804

In [86]:
# ==========================================
# CELL 14: LOAD TRAINED TEACHER
# teacher is MobileNetV3Large transfer learning model
# ==========================================
teacher = load_model(teacher_model_path)
teacher.trainable = False

print("Teacher loaded successfully.")
teacher.summary()

Teacher loaded successfully.


Model: "teacher_mobilenetv3_hydrophobicity"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 960)            │         3,840 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       246,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,749,531 (14.30 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,249,031 (12.39 MB)

 Optimizer params: 500,500 (1.91 MB)

In [87]:
# ==========================================
# CELL 15: KD DUAL GENERATOR
# student branch = 32x32 + preprocess_input
# teacher branch = 224x224 + preprocess_input
# ==========================================
class KDDualSequence(tf.keras.utils.Sequence):
    def __init__(self, directory, batch_size, student_size, teacher_size, shuffle, seed=None):
        self.batch_size = batch_size
        self.student_size = student_size
        self.teacher_size = teacher_size
        self.shuffle = shuffle

        self.base_gen = ImageDataGenerator()
        self.flow = self.base_gen.flow_from_directory(
            directory,
            target_size=(teacher_size, teacher_size),
            batch_size=batch_size,
            class_mode='categorical',
            shuffle=shuffle,
            seed=seed
        )

    def __len__(self):
        return len(self.flow)

    def __getitem__(self, idx):
        images_teacher_raw, labels = self.flow[idx]

        student_images = tf.image.resize(images_teacher_raw, (self.student_size, self.student_size)).numpy()

        student_images = preprocess_input(student_images.copy())
        teacher_images = preprocess_input(images_teacher_raw.copy())

        return {
            "student_input": student_images,
            "teacher_input": teacher_images
        }, labels

    def on_epoch_end(self):
        self.flow.on_epoch_end()

In [88]:
# ==========================================
# CELL 16: CREATE KD DATASETS
# ==========================================
train_kd_ds = KDDualSequence(
    directory=train_path,
    batch_size=batch_size,
    student_size=student_img_size,
    teacher_size=teacher_img_size,
    shuffle=True,
    seed=SEED
)

val_kd_ds = KDDualSequence(
    directory=val_path,
    batch_size=batch_size,
    student_size=student_img_size,
    teacher_size=teacher_img_size,
    shuffle=False
)

print("KD datasets ready.")
print("Student input : 32x32 + external MobileNet preprocessing")
print("Teacher input : 224x224 + external MobileNet preprocessing")

Found 2800 images belonging to 7 classes.
Found 700 images belonging to 7 classes.
KD datasets ready.
Student input : 32x32 + external MobileNet preprocessing
Teacher input : 224x224 + external MobileNet preprocessing


In [89]:
# ==========================================
# CELL 17: TEMPERATURE SCALING HELPER
# ==========================================
def temp_scale_probs(p, T, eps=1e-8):
    p = tf.clip_by_value(p, eps, 1.0)
    pT = tf.pow(p, 1.0 / T)
    pT = pT / tf.reduce_sum(pT, axis=-1, keepdims=True)
    return pT

In [90]:
# ==========================================
# CELL 18: SAFE DISTILLER
# ==========================================
class SafeDistiller(keras.Model):
    def __init__(self, student, teacher, T=5.0, alpha=0.3):
        super().__init__(name="safe_distiller_dual_input")
        self.student = student
        self.teacher = teacher
        self.teacher.trainable = False

        self.T = float(T)
        self.alpha = float(alpha)

        self.ce = keras.losses.CategoricalCrossentropy()
        self.kld = keras.losses.KLDivergence()

        self.acc = keras.metrics.CategoricalAccuracy(name="acc")
        self.hard_tracker = keras.metrics.Mean(name="hard_loss")
        self.soft_tracker = keras.metrics.Mean(name="soft_loss")
        self.kd_tracker = keras.metrics.Mean(name="kd_loss")

    @property
    def metrics(self):
        return [self.acc, self.hard_tracker, self.soft_tracker, self.kd_tracker]

    def compile(self, optimizer, **kwargs):
        super().compile(optimizer=optimizer, **kwargs)

    def call(self, inputs, training=False):
        if isinstance(inputs, dict):
            student_x = inputs["student_input"]
        else:
            student_x = inputs
        return self.student(student_x, training=training)

    def train_step(self, data):
        x, y = data
        student_x = x["student_input"]
        teacher_x = x["teacher_input"]

        teacher_probs = self.teacher(teacher_x, training=False)

        with tf.GradientTape() as tape:
            student_probs = self.student(student_x, training=True)

            hard_loss = self.ce(y, student_probs)

            teacher_T = temp_scale_probs(teacher_probs, self.T)
            student_T = temp_scale_probs(student_probs, self.T)
            soft_loss = self.kld(teacher_T, student_T)

            kd_total = self.alpha * hard_loss + (1.0 - self.alpha) * (self.T ** 2) * soft_loss

        grads = tape.gradient(kd_total, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))

        self.acc.update_state(y, student_probs)
        self.hard_tracker.update_state(hard_loss)
        self.soft_tracker.update_state(soft_loss)
        self.kd_tracker.update_state(kd_total)

        return {
            "loss": self.kd_tracker.result(),
            "acc": self.acc.result(),
            "hard_loss": self.hard_tracker.result(),
            "soft_loss": self.soft_tracker.result(),
            "kd_loss": self.kd_tracker.result()
        }

    def test_step(self, data):
        x, y = data
        student_x = x["student_input"]

        student_probs = self.student(student_x, training=False)
        loss = self.ce(y, student_probs)

        self.acc.update_state(y, student_probs)
        self.hard_tracker.update_state(loss)

        return {
            "loss": loss,
            "acc": self.acc.result(),
            "hard_loss": self.hard_tracker.result()
        }

In [91]:
# ==========================================
# CELL 19: CUSTOM CHECKPOINT FOR KD STUDENT ONLY
# ==========================================
class StudentCheckpoint(keras.callbacks.Callback):
    def __init__(self, filepath, monitor="val_acc", mode="max", verbose=1):
        super().__init__()
        self.filepath = filepath
        self.monitor = monitor
        self.mode = mode
        self.verbose = verbose
        self.best = -np.inf if mode == "max" else np.inf

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)

        if current is None:
            if self.verbose:
                print(f"Metric {self.monitor} not found in logs. Available keys: {list(logs.keys())}")
            return

        improved = current > self.best if self.mode == "max" else current < self.best

        if improved:
            self.best = current
            self.model.student.save(self.filepath)
            if self.verbose:
                print(f"Epoch {epoch+1}: {self.monitor} improved to {current:.4f}, saved student to {self.filepath}")

In [92]:
# ==========================================
# CELL 20: KD HYPERPARAMETERS
# ==========================================
T = 5.0
alpha = 0.3

print("KD settings")
print("Temperature:", T)
print("Alpha:", alpha)

KD settings
Temperature: 5.0
Alpha: 0.3


In [93]:
# ==========================================
# CELL 21: KD CALLBACKS
# ==========================================
checkpoint_kd_student = StudentCheckpoint(
    filepath=kd_model_path,
    monitor="val_acc",
    mode="max",
    verbose=1
)

lr_reducer_kd = keras.callbacks.ReduceLROnPlateau(
    monitor="val_acc",
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=1,
    mode="max"
)

early_stop_kd = keras.callbacks.EarlyStopping(
    monitor="val_acc",
    patience=7,
    restore_best_weights=True,
    verbose=1,
    mode="max"
)

In [94]:
# ==========================================
# CELL 22: BUILD STUDENT FOR KD
# ==========================================
student_kd = create_mobilenetv3_student(
    input_shape=(student_img_size, student_img_size, 3),
    num_classes=num_classes,
    **baseline_params
)

student_kd.summary()

Model: "student_mobilenetv3_hydrophobicity"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_57 (Conv2D)  │ (None, 32, 32,    │        864 │ input_layer_9[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ conv2d_57[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_55 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_58 (Conv2D)  │ (None, 32, 32,    │      4,096 │ re_lu_55[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_58[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_56 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d_24 │ (None, 16, 16,    │      3,200 │ re_lu_56[0][0]    │
│ (DepthwiseConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_57 (ReLU)     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ re_lu_57[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_62 (Dense)    │ (None, 32)        │      4,128 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_63 (Dense)    │ (None, 128)       │      4,224 │ dense_62[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_24          │ (None, 1, 1, 128) │          0 │ dense_63[0][0]    │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_24         │ (None, 16, 16,    │          0 │ re_lu_57[0][0],   │
│ (Multiply)          │ 128)              │            │ reshape_24[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_59 (Conv2D)  │ (None, 16, 16,    │      8,192 │ multiply_24[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_59[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,089,223 (15.60 MB)

 Trainable params: 4,067,399 (15.52 MB)

 Non-trainable params: 21,824 (85.25 KB)

In [95]:
# ==========================================
# CELL 23: TRAIN KD STUDENT
# ==========================================
print("Training KD student on hydrophobicity...")

distiller = SafeDistiller(
    student=student_kd,
    teacher=teacher,
    T=T,
    alpha=alpha
)

distiller.compile(
    optimizer=keras.optimizers.Adam(1e-3)
)

history_kd = distiller.fit(
    train_kd_ds,
    validation_data=val_kd_ds,
    epochs=epochs,
    callbacks=[early_stop_kd, checkpoint_kd_student, lr_reducer_kd],
    verbose=2
)

student_kd.save(kd_model_path)
print("KD student saved.")

Training KD student on hydrophobicity...
Epoch 1/20


2026-04-30 18:32:50.688272: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_broadcast_multiply_reduce_fusion', 4 bytes spill stores, 4 bytes spill loads



Epoch 1: val_acc improved to 0.1429, saved student to hydro_kd_M5.keras
88/88 - 96s - 1s/step - acc: 0.6646 - hard_loss: 1.2208 - kd_loss: 5.3614 - loss: 5.3614 - soft_loss: 0.2854 - val_acc: 0.1429 - val_hard_loss: 2.0705 - val_loss: 1.8964 - learning_rate: 0.0010
Epoch 2/20
88/88 - 9s - 108ms/step - acc: 0.8007 - hard_loss: 0.7053 - kd_loss: 3.0198 - loss: 3.0198 - soft_loss: 0.1605 - val_acc: 0.1429 - val_hard_loss: 2.3656 - val_loss: 2.2662 - learning_rate: 0.0010
Epoch 3/20
88/88 - 10s - 111ms/step - acc: 0.8771 - hard_loss: 0.4331 - kd_loss: 2.0071 - loss: 2.0071 - soft_loss: 0.1073 - val_acc: 0.1429 - val_hard_loss: 2.6586 - val_loss: 2.9010 - learning_rate: 0.0010
Epoch 4/20
88/88 - 10s - 110ms/step - acc: 0.9043 - hard_loss: 0.3082 - kd_loss: 1.4527 - loss: 1.4527 - soft_loss: 0.0777 - val_acc: 0.1429 - val_hard_loss: 3.4870 - val_loss: 4.0415 - learning_rate: 0.0010
Epoch 5/20
Epoch 5: val_acc improved to 0.2300, saved student to hydro_kd_M5.keras
88/88 - 10s - 113ms/step - a

In [96]:
# ==========================================
# CELL 24: FINAL EVALUATION
# baseline and KD are both evaluated on 32x32 student validation data
# ==========================================
print("Baseline Student Evaluation")
baseline_loss, baseline_acc = student_baseline.evaluate(val_data_student32, verbose=1)
print(f"Baseline val loss: {baseline_loss:.4f}")
print(f"Baseline val acc : {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")

print("\nKD Student Evaluation")
kd_loss, kd_acc = student_kd.evaluate(val_data_student32, verbose=1)
print(f"KD val loss: {kd_loss:.4f}")
print(f"KD val acc : {kd_acc:.4f} ({kd_acc*100:.2f}%)")

Baseline Student Evaluation
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.7971 - loss: 0.4955
Baseline val loss: 0.4955
Baseline val acc : 0.7971 (79.71%)

KD Student Evaluation
22/22 ━━━━━━━━━━━━━━━━━━━━ 10s 225ms/step - accuracy: 0.9300 - loss: 0.2966
KD val loss: 0.2966
KD val acc : 0.9300 (93.00%)


In [35]:
# ==========================================
# CELL 25: PARAMETER COUNT
# ==========================================
print("Baseline params:", student_baseline.count_params())
print("KD student params:", student_kd.count_params())

Baseline params: 26471
KD student params: 26471


In [56]:
# ==========================================
# CELL 26: MODEL FILE SIZES
# ==========================================
for path in [baseline_model_path, kd_model_path, teacher_model_path]:
    if os.path.exists(path):
        print(f"{path} - {os.path.getsize(path)/(1024**2):.2f} MB")
    else:
        print(f"{path} not found")

hydro_baseline_M3.keras - 2.29 MB
hydro_kd_M3.keras - 0.55 MB
/home/22EC1102/soumen/satarupa/hydrophobicity/teacher_model_mobilenetv3.keras - 14.98 MB
